In [ ]:
import matplotlib
import numpy
import os
import pandas
import random
import sys
import torch
import transformers

In [ ]:
def set_all_seeds(seed):
    random.seed(seed)
    numpy.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    transformers.set_seed(seed)

set_all_seeds(42)

In [ ]:
model_name = "pborchert/BusinessBERT"
model = transformers.BertForMultipleChoice.from_pretrained(model_name)
tokenizer = transformers.BertTokenizer.from_pretrained(model_name)

In [ ]:
# Comment out all but the appropriate option
where_running = "Google Colab"
# where_running = "Local Machine"

if where_running == "Local Machine":
  module_path = os.path.abspath(os.path.join('..'))
  if not module_path in sys.path:
      sys.path.insert(0, module_path)
elif where_running == "Google Colab":
  dirpath = '/content/digi-inno-road-prod'
  if not os.path.isdir(dirpath):
    # TODO git pull
    !git clone https://github.com/roughhawkbit/digi-inno-road-prod.git
  sys.path.insert(0,dirpath)

from innoprod.plotting_tools import rand_jitter
from innoprod.sheet_tools import get_sheet_dfs
from innoprod.wrangling.msyh_data_sharing import wrangle_roadmaps
from innoprod.wrangling.wrangling_tools import is_non_empty

In [ ]:
data = get_sheet_dfs()
roadmaps_df = wrangle_roadmaps(data['Roadmaps'])

In [ ]:
drs_data = roadmaps_df[[
    'Client ID',
    'Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples',
    'Current Digital Readiness Score (refer to PAS:1040)',
    ]].rename(columns={
        'Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples': 'Context',
        'Current Digital Readiness Score (refer to PAS:1040)': 'DRS',
    })

drs_data = drs_data[drs_data['DRS'].notna() & is_non_empty(drs_data['Context'])]

In [ ]:
prompt = drs_data['Context'].to_list()[0]
prompt

The DRS levels from [PAS 1040:2019 Digital Technologies in Manufacturing](https://www.bsigroup.com/en-GB/insights-and-media/insights/brochures/pas-1040-adopting-digital-technologies-in-manufacturing-guide/) Table 6.

In [ ]:
choices = [
    "The business has no vision for driving growth with digital technologies, and is not supporting workers to investigate opportunities.", # 1
    "Workers are aware of the potential for digital technologies and are supported by the business to experiment with local trials.", # 2
    "The business is learning from local trials of digital technologies and leaders are investigating the business case.", # 3
    "The business has a strategic vision for digital transformation, the business case is agreed and implementation is underway.", # 4
    "Workers are engaged in digital transformation and the business is starting to achieve business case benefits.", # 5
    "Digital technologies are driving continuous improvement in key aspects of operational performance including supply chain and customer services.", # 6
    "Innovation with digital technologies is part of the culture of the business and is driving continuous improvement in all aspects.", # 7
    "Increasing adoption of digital technologies is sustained by reinvestment of related profits and continuous renewal of the business case.", # 8
    "Digital technologies are driving optimized productivity and competitiveness for the business and its partners.", # 9
]


In [ ]:
def calc_choice_scores(context):
  encoding = tokenizer([context]*len(choices), choices, return_tensors='pt', padding=True)
  outputs = model(**{k: v.unsqueeze(0) for k,v in encoding.items()})
  return outputs.logits.tolist()[0]


Running over the whole dataset takes about 10 minutes. Uncomment the code cell below to analyse just 10 rows in about half a minute.

In [ ]:
# drs_data = drs_data[:10].copy(deep=True)
# len(drs_data)

In [ ]:
drs_data['scores'] = drs_data['Context'].apply(calc_choice_scores)

In [ ]:
for i in range(len(choices)):
  drs_data[f'DRS_{i+1}'] = drs_data['scores'].apply(lambda x: x[i])
drs_data['DRS_pred'] = drs_data['scores'].apply(lambda x: x.index(max(x))+1)


In [ ]:
x = drs_data['DRS'].values.reshape(-1, 1)
y = drs_data['DRS_pred'].values.reshape(-1, 1)

fig, ax = matplotlib.pyplot.subplots(figsize=(6,6))

stdev = 0.2
ax.scatter(
    rand_jitter(x, stdev=stdev),
    rand_jitter(y, stdev=stdev),
    c=((drs_data['DRS'].values+drs_data['DRS_pred'].values)%2),
    zorder=0
  )

ax.plot([1,9], [1,9], zorder=1, c='r')

ax.set_xlabel('DRS')
ax.set_ylabel('BERT-predicted DRS')

In [ ]:
drs_data['DRS'].value_counts().sort_index().plot(kind="bar")

In [ ]:
drs_data['DRS_pred'].value_counts().sort_index().plot(kind="bar")